У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [7]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/customer_segmentation_train.csv')

df = df.drop('ID', axis=1)
X = df.drop('Segmentation', axis=1)
y = df['Segmentation']

le = LabelEncoder()
y = le.fit_transform(y)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"train: {X_train_processed.shape}")
print(f"test: {X_test_processed.shape}")

train: (6454, 28)
test: (1614, 28)


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [21]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

X_train_imputed = X_train.copy()
X_train_imputed[numeric_features] = num_imputer.fit_transform(X_train[numeric_features])
X_train_imputed[categorical_features] = cat_imputer.fit_transform(X_train[categorical_features])

ordinal_enc = OrdinalEncoder()
X_train_encoded = X_train_imputed.copy()
X_train_encoded[categorical_features] = ordinal_enc.fit_transform(X_train_imputed[categorical_features])

cat_feature_indices = [X_train_imputed.columns.get_loc(col) for col in categorical_features]

print(f"Індекси кат. ознак: {cat_feature_indices}")

Індекси кат. ознак: [0, 1, 3, 4, 6, 8]


In [22]:
smotenc = SMOTENC(categorical_features=cat_feature_indices, random_state=42)
X_train_smote_ord, y_train_smote = smotenc.fit_resample(X_train_encoded, y_train)

smote_tomek = SMOTETomek(smote=smotenc, random_state=42)
X_train_tomek_ord, y_train_tomek = smote_tomek.fit_resample(X_train_encoded, y_train)

In [29]:
display(X_train_tomek_ord.shape)
display(pd.Series(y_train_tomek).value_counts().sort_index().tolist())

(6096, 9)

[1473, 1507, 1554, 1562]

In [30]:
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']

def final_preprocess(df_ord, numeric_cols, categorical_cols, encoder):
    num_data = df_ord[numeric_cols].values
    cat_data_raw = ordinal_enc.inverse_transform(df_ord[categorical_cols])
    cat_data_ohe = encoder.transform(cat_data_raw)
    return np.hstack([num_data, cat_data_ohe])

X_train_smote_final = final_preprocess(X_train_smote_ord, numeric_features, categorical_features, ohe)

X_train_tomek_final = final_preprocess(X_train_tomek_ord, numeric_features, categorical_features, ohe)

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [32]:
datasets = {
    "Оригінальні дані": (X_train_processed, y_train),
    "SMOTE (SMOTENC)": (X_train_smote_final, y_train_smote),
    "SMOTE-Tomek": (X_train_tomek_final, y_train_tomek)
}

for name, (X_tr, y_tr) in datasets.items():
    print(f"Модель: {name}")
    model = LogisticRegression(multi_class='ovr', solver='liblinear', random_state=42, max_iter=1000)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_test_processed)
    print(classification_report(y_test, y_pred, target_names=le.classes_))

Модель: Оригінальні дані
              precision    recall  f1-score   support

           A       0.42      0.45      0.43       394
           B       0.41      0.18      0.25       372
           C       0.49      0.61      0.54       394
           D       0.65      0.76      0.70       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.51      0.49      1614

Модель: SMOTE (SMOTENC)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

           A       0.35      0.42      0.38       394
           B       0.26      0.16      0.20       372
           C       0.88      0.02      0.03       394
           D       0.45      0.89      0.60       454

    accuracy                           0.40      1614
   macro avg       0.48      0.37      0.30      1614
weighted avg       0.48      0.40      0.32      1614

Модель: SMOTE-Tomek
              precision    recall  f1-score   support

           A       0.36      0.42      0.39       394
           B       0.27      0.20      0.23       372
           C       0.75      0.01      0.02       394
           D       0.46      0.89      0.61       454

    accuracy                           0.40      1614
   macro avg       0.46      0.38      0.31      1614
weighted avg       0.46      0.40      0.32      1614



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Для порівняння моделей обрав Macro F1-score.
Найкращою є модель, яка навчена на оригінальних даних. Має найвищу accuracy.